Synthetic data generation for Life & Health Insurance Underwriting business
*Co-authored with CoCo*

In [ ]:
%%sql -r setup_result
CREATE DATABASE IF NOT EXISTS INSURANCE_UNDERWRITING;
USE DATABASE INSURANCE_UNDERWRITING;
CREATE SCHEMA IF NOT EXISTS RAW;
USE SCHEMA RAW;

In [ ]:
%%sql -r create_policyholders
CREATE OR REPLACE TABLE RAW_POLICYHOLDERS (
    POLICYHOLDER_ID     VARCHAR(20)     PRIMARY KEY,
    FIRST_NAME          VARCHAR(50),
    LAST_NAME           VARCHAR(50),
    DATE_OF_BIRTH       DATE,
    GENDER              VARCHAR(10),
    MARITAL_STATUS      VARCHAR(20),
    EMAIL               VARCHAR(100),
    PHONE               VARCHAR(20),
    ADDRESS_LINE_1      VARCHAR(200),
    CITY                VARCHAR(50),
    STATE_CODE          VARCHAR(2),
    ZIP_CODE            VARCHAR(10),
    OCCUPATION          VARCHAR(100),
    ANNUAL_INCOME       NUMBER(12,2),
    SMOKER_FLAG         BOOLEAN,
    BMI                 NUMBER(4,1),
    PRE_EXISTING_CONDITIONS VARCHAR(500),
    CREATED_AT          TIMESTAMP_NTZ,
    UPDATED_AT          TIMESTAMP_NTZ
);

In [ ]:
%%sql -r insert_policyholders
INSERT INTO RAW_POLICYHOLDERS
WITH
name_arrays AS (
    SELECT
        ARRAY_CONSTRUCT('James','John','Robert','Michael','David','William','Richard','Joseph','Thomas','Charles',
                        'Daniel','Matthew','Anthony','Mark','Steven','Andrew','Paul','Brian','Kevin','Jason',
                        'Mary','Patricia','Jennifer','Linda','Barbara','Elizabeth','Susan','Jessica','Sarah','Karen',
                        'Lisa','Nancy','Betty','Margaret','Sandra','Ashley','Dorothy','Kimberly','Emily','Donna') AS first_names,
        ARRAY_CONSTRUCT('M','M','M','M','M','M','M','M','M','M','M','M','M','M','M','M','M','M','M','M',
                        'F','F','F','F','F','F','F','F','F','F','F','F','F','F','F','F','F','F','F','F') AS genders,
        ARRAY_CONSTRUCT('Smith','Johnson','Williams','Brown','Jones','Garcia','Miller','Davis','Rodriguez','Martinez',
                        'Hernandez','Lopez','Gonzalez','Wilson','Anderson','Thomas','Taylor','Moore','Jackson','Martin',
                        'Lee','Perez','Thompson','White','Harris','Sanchez','Clark','Ramirez','Lewis','Robinson',
                        'Walker','Young','Allen','King','Wright','Scott','Torres','Nguyen','Hill','Flores') AS last_names,
        ARRAY_CONSTRUCT('Los Angeles','San Francisco','San Diego','Houston','Dallas','Austin','New York','Buffalo',
                        'Miami','Orlando','Tampa','Chicago','Philadelphia','Columbus','Atlanta','Charlotte','Detroit',
                        'Newark','Richmond','Seattle','Boston','Phoenix','Denver','Nashville','Minneapolis') AS cities,
        ARRAY_CONSTRUCT('CA','CA','CA','TX','TX','TX','NY','NY','FL','FL','FL','IL','PA','OH','GA','NC','MI',
                        'NJ','VA','WA','MA','AZ','CO','TN','MN') AS state_codes,
        ARRAY_CONSTRUCT('Software Engineer','Teacher','Nurse','Accountant','Sales Manager','Attorney','Physician',
                        'Construction Worker','Retail Associate','Marketing Director','Financial Analyst','Truck Driver',
                        'Electrician','Pharmacist','Police Officer','Firefighter','Chef','Architect',
                        'Real Estate Agent','Dental Hygienist') AS occupations,
        ARRAY_CONSTRUCT('None','None','None','None','None','None','None','Hypertension','Diabetes Type 2','Asthma',
                        'High Cholesterol','Arthritis','Anxiety Disorder','Depression','Heart Disease',
                        'Thyroid Disorder','COPD','Sleep Apnea') AS conditions
),
row_gen AS (
    SELECT SEQ4() + 1 AS rn, RANDOM() AS r1, RANDOM() AS r2, RANDOM() AS r3, RANDOM() AS r4,
           RANDOM() AS r5, RANDOM() AS r6, RANDOM() AS r7, RANDOM() AS r8, RANDOM() AS r9
    FROM TABLE(GENERATOR(ROWCOUNT => 1500))
)
SELECT
    'PH-' || LPAD(rg.rn::VARCHAR, 6, '0'),
    na.first_names[ABS(MOD(rg.r1, 40))]::VARCHAR,
    na.last_names[ABS(MOD(rg.r2, 40))]::VARCHAR,
    DATEADD('day', -ABS(MOD(rg.r3, 18250)) - 7300, CURRENT_DATE()),
    na.genders[ABS(MOD(rg.r1, 40))]::VARCHAR,
    CASE ABS(MOD(rg.r4, 4))
        WHEN 0 THEN 'Single' WHEN 1 THEN 'Married' WHEN 2 THEN 'Divorced' ELSE 'Widowed'
    END,
    LOWER(na.first_names[ABS(MOD(rg.r1, 40))]::VARCHAR || '.' ||
          na.last_names[ABS(MOD(rg.r2, 40))]::VARCHAR ||
          ABS(MOD(rg.r5, 999))::VARCHAR || '@email.com'),
    '(' || LPAD(ABS(MOD(rg.r6, 800) + 200)::VARCHAR, 3, '0') || ') ' ||
        LPAD(ABS(MOD(rg.r7, 900) + 100)::VARCHAR, 3, '0') || '-' ||
        LPAD(ABS(MOD(rg.r8, 9000) + 1000)::VARCHAR, 4, '0'),
    ABS(MOD(rg.r5, 9900) + 100)::VARCHAR || ' ' ||
        CASE ABS(MOD(rg.r6, 5)) WHEN 0 THEN 'Oak' WHEN 1 THEN 'Maple' WHEN 2 THEN 'Elm' WHEN 3 THEN 'Pine' ELSE 'Cedar' END || ' ' ||
        CASE ABS(MOD(rg.r7, 4)) WHEN 0 THEN 'St' WHEN 1 THEN 'Ave' WHEN 2 THEN 'Blvd' ELSE 'Dr' END,
    na.cities[ABS(MOD(rg.r3, 25))]::VARCHAR,
    na.state_codes[ABS(MOD(rg.r3, 25))]::VARCHAR,
    LPAD(ABS(MOD(rg.r9, 90000) + 10000)::VARCHAR, 5, '0'),
    na.occupations[ABS(MOD(rg.r4, 20))]::VARCHAR,
    ROUND(ABS(MOD(rg.r5, 225000)) + 25000 + ABS(MOD(rg.r6, 100)) / 100.0, 2),
    CASE WHEN ABS(MOD(rg.r7, 100)) < 18 THEN TRUE ELSE FALSE END,
    ROUND((ABS(MOD(rg.r8, 240)) + 180) / 10.0, 1),
    na.conditions[ABS(MOD(rg.r9, 18))]::VARCHAR,
    DATEADD('day', -ABS(MOD(rg.r1, 1795)) - 30, CURRENT_TIMESTAMP()),
    DATEADD('day', -ABS(MOD(rg.r2, 29)), CURRENT_TIMESTAMP())
FROM row_gen rg
CROSS JOIN name_arrays na;

In [ ]:
%%sql -r create_policies
CREATE OR REPLACE TABLE RAW_POLICIES (
    POLICY_ID           VARCHAR(20)     PRIMARY KEY,
    POLICYHOLDER_ID     VARCHAR(20),
    PRODUCT_LINE        VARCHAR(50),
    PRODUCT_NAME        VARCHAR(100),
    POLICY_STATUS       VARCHAR(20),
    EFFECTIVE_DATE      DATE,
    EXPIRATION_DATE     DATE,
    TERM_YEARS          NUMBER(2),
    SUM_ASSURED         NUMBER(14,2),
    ANNUAL_PREMIUM      NUMBER(10,2),
    PAYMENT_FREQUENCY   VARCHAR(20),
    BENEFICIARY_NAME    VARCHAR(100),
    BENEFICIARY_RELATIONSHIP VARCHAR(50),
    AGENT_ID            VARCHAR(20),
    APPLICATION_DATE    DATE,
    ISSUE_DATE          DATE,
    CREATED_AT          TIMESTAMP_NTZ,
    UPDATED_AT          TIMESTAMP_NTZ
);

In [ ]:
%%sql -r insert_policies
INSERT INTO RAW_POLICIES
WITH
prod_arrays AS (
    SELECT
        ARRAY_CONSTRUCT('Term Life','Term Life','Term Life','Whole Life','Whole Life',
                        'Health','Health','Health','Disability','Disability',
                        'Critical Illness','Critical Illness') AS product_lines,
        ARRAY_CONSTRUCT('10-Year Term Life','20-Year Term Life','30-Year Term Life',
                        'Standard Whole Life','Universal Life',
                        'Individual Health Plan','Family Health Plan','High Deductible Health',
                        'Short-Term Disability','Long-Term Disability',
                        'Cancer Protection','Heart & Stroke Cover') AS product_names,
        ARRAY_CONSTRUCT(50000,100000,100000,25000,50000,0,0,0,0,0,25000,25000) AS min_sums,
        ARRAY_CONSTRUCT(2000000,5000000,3000000,1000000,2000000,500000,1000000,300000,150000,500000,500000,500000) AS max_sums,
        ARRAY_CONSTRUCT(10,20,30,99,99,3,3,3,2,5,3,3) AS term_years_arr
),
row_gen AS (
    SELECT SEQ4() + 1 AS rn, RANDOM() AS r1, RANDOM() AS r2, RANDOM() AS r3,
           RANDOM() AS r4, RANDOM() AS r5, RANDOM() AS r6, RANDOM() AS r7,
           RANDOM() AS r8, RANDOM() AS r9
    FROM TABLE(GENERATOR(ROWCOUNT => 3000))
)
SELECT
    'POL-' || LPAD(rg.rn::VARCHAR, 7, '0'),
    'PH-' || LPAD((ABS(MOD(rg.r1, 1500)) + 1)::VARCHAR, 6, '0'),
    pa.product_lines[ABS(MOD(rg.r2, 12))]::VARCHAR,
    pa.product_names[ABS(MOD(rg.r2, 12))]::VARCHAR,
    CASE
        WHEN ABS(MOD(rg.r3, 100)) < 3 THEN 'Pending'
        WHEN ABS(MOD(rg.r3, 100)) < 5 THEN 'Cancelled'
        WHEN ABS(MOD(rg.r3, 100)) < 80 THEN 'Active'
        WHEN ABS(MOD(rg.r3, 100)) < 93 THEN 'Lapsed'
        ELSE 'Expired'
    END,
    DATEADD('day', -(ABS(MOD(rg.r4, 1770)) + 30), CURRENT_DATE()),
    DATEADD('year', pa.term_years_arr[ABS(MOD(rg.r2, 12))]::INTEGER,
            DATEADD('day', -(ABS(MOD(rg.r4, 1770)) + 30), CURRENT_DATE())),
    pa.term_years_arr[ABS(MOD(rg.r2, 12))]::INTEGER,
    ROUND((ABS(MOD(rg.r5, (pa.max_sums[ABS(MOD(rg.r2, 12))]::INTEGER - pa.min_sums[ABS(MOD(rg.r2, 12))]::INTEGER + 1)))
           + pa.min_sums[ABS(MOD(rg.r2, 12))]::INTEGER)::NUMBER(14,2), -3),
    ROUND(ABS(MOD(rg.r6, 14500)) + 500 + ABS(MOD(rg.r7, 100)) / 100.0, 2),
    CASE ABS(MOD(rg.r8, 4))
        WHEN 0 THEN 'Monthly' WHEN 1 THEN 'Quarterly' WHEN 2 THEN 'Semi-Annual' ELSE 'Annual'
    END,
    CASE ABS(MOD(rg.r9, 8))
        WHEN 0 THEN 'Jane' WHEN 1 THEN 'John' WHEN 2 THEN 'Sarah' WHEN 3 THEN 'Michael'
        WHEN 4 THEN 'Emily' WHEN 5 THEN 'David' WHEN 6 THEN 'Maria' ELSE 'Robert'
    END || ' ' ||
    CASE ABS(MOD(rg.r1, 6))
        WHEN 0 THEN 'Smith' WHEN 1 THEN 'Johnson' WHEN 2 THEN 'Williams'
        WHEN 3 THEN 'Brown' WHEN 4 THEN 'Davis' ELSE 'Wilson'
    END,
    CASE ABS(MOD(rg.r2, 6))
        WHEN 0 THEN 'Spouse' WHEN 1 THEN 'Child' WHEN 2 THEN 'Parent'
        WHEN 3 THEN 'Sibling' WHEN 4 THEN 'Estate' ELSE 'Trust'
    END,
    'AGT-' || LPAD((ABS(MOD(rg.r3, 200)) + 1)::VARCHAR, 4, '0'),
    DATEADD('day', -(ABS(MOD(rg.r4, 1770)) + 60), CURRENT_DATE()),
    DATEADD('day', -(ABS(MOD(rg.r4, 1770)) + 30), CURRENT_DATE()),
    DATEADD('day', -(ABS(MOD(rg.r5, 1770)) + 30), CURRENT_TIMESTAMP()),
    DATEADD('day', -ABS(MOD(rg.r6, 29)), CURRENT_TIMESTAMP())
FROM row_gen rg
CROSS JOIN prod_arrays pa;

In [ ]:
%%sql -r create_uw_decisions
CREATE OR REPLACE TABLE RAW_UNDERWRITING_DECISIONS (
    DECISION_ID         VARCHAR(20)     PRIMARY KEY,
    POLICY_ID           VARCHAR(20),
    POLICYHOLDER_ID     VARCHAR(20),
    UNDERWRITER_ID      VARCHAR(20),
    DECISION_DATE       DATE,
    RISK_CLASS          VARCHAR(30),
    RISK_SCORE          NUMBER(5,2),
    DECISION_OUTCOME    VARCHAR(30),
    DECLINE_REASON      VARCHAR(200),
    PREMIUM_LOADING_PCT NUMBER(5,2),
    EXCLUSIONS_APPLIED  VARCHAR(500),
    MEDICAL_EXAM_REQUIRED BOOLEAN,
    MEDICAL_EXAM_DATE   DATE,
    MEDICAL_EXAM_RESULT VARCHAR(50),
    NOTES               VARCHAR(1000),
    CREATED_AT          TIMESTAMP_NTZ
);

In [ ]:
%%sql -r insert_uw_decisions
INSERT INTO RAW_UNDERWRITING_DECISIONS
SELECT
    'UW-' || LPAD(ROW_NUMBER() OVER (ORDER BY POLICY_ID)::VARCHAR, 7, '0'),
    POLICY_ID,
    POLICYHOLDER_ID,
    'UWR-' || LPAD((ABS(MOD(RANDOM(), 50)) + 1)::VARCHAR, 4, '0'),
    DATEADD('day', ABS(MOD(RANDOM(), 18)) + 3, APPLICATION_DATE),
    CASE
        WHEN ABS(MOD(RANDOM(), 100)) < 3 THEN 'Super Preferred'
        WHEN ABS(MOD(RANDOM(), 100)) < 23 THEN 'Preferred Plus'
        WHEN ABS(MOD(RANDOM(), 100)) < 53 THEN 'Preferred'
        WHEN ABS(MOD(RANDOM(), 100)) < 83 THEN 'Standard'
        WHEN ABS(MOD(RANDOM(), 100)) < 93 THEN 'Substandard'
        ELSE 'Decline'
    END,
    ROUND(ABS(MOD(RANDOM(), 850) + 100) / 10.0, 2),
    CASE
        WHEN ABS(MOD(RANDOM(), 100)) < 70 THEN 'Approved - Standard'
        WHEN ABS(MOD(RANDOM(), 100)) < 85 THEN 'Approved - Rated'
        WHEN ABS(MOD(RANDOM(), 100)) < 92 THEN 'Approved - Exclusion'
        WHEN ABS(MOD(RANDOM(), 100)) < 95 THEN 'Deferred'
        ELSE 'Declined'
    END,
    CASE WHEN ABS(MOD(RANDOM(), 100)) >= 95 THEN
        CASE ABS(MOD(RANDOM(), 5))
            WHEN 0 THEN 'Excessive mortality risk'
            WHEN 1 THEN 'Undisclosed medical history'
            WHEN 2 THEN 'High-risk occupation'
            WHEN 3 THEN 'Failed medical examination'
            ELSE 'Incomplete application'
        END
        ELSE NULL
    END,
    CASE WHEN ABS(MOD(RANDOM(), 100)) BETWEEN 70 AND 84
         THEN ROUND(ABS(MOD(RANDOM(), 175)) + 25, 2)
         ELSE 0
    END,
    CASE WHEN ABS(MOD(RANDOM(), 100)) BETWEEN 85 AND 91 THEN
        CASE ABS(MOD(RANDOM(), 4))
            WHEN 0 THEN 'Pre-existing cardiac conditions excluded'
            WHEN 1 THEN 'Mental health conditions excluded for first 2 years'
            WHEN 2 THEN 'Hazardous sports activities excluded'
            ELSE 'Cancer coverage excluded for first 12 months'
        END
        ELSE NULL
    END,
    CASE WHEN ABS(MOD(RANDOM(), 100)) < 60 THEN TRUE ELSE FALSE END,
    CASE WHEN ABS(MOD(RANDOM(), 100)) < 60
         THEN DATEADD('day', ABS(MOD(RANDOM(), 10)) + 5, APPLICATION_DATE)
         ELSE NULL
    END,
    CASE WHEN ABS(MOD(RANDOM(), 100)) < 60 THEN
        CASE ABS(MOD(RANDOM(), 4))
            WHEN 0 THEN 'Normal' WHEN 1 THEN 'Normal'
            WHEN 2 THEN 'Minor Issues Noted' ELSE 'Significant Findings'
        END
        ELSE NULL
    END,
    CASE ABS(MOD(RANDOM(), 6))
        WHEN 0 THEN 'Standard underwriting process completed'
        WHEN 1 THEN 'Applicant has family history of cardiac disease'
        WHEN 2 THEN 'BMI slightly elevated but within acceptable range'
        WHEN 3 THEN 'Occupation risk assessed as moderate'
        WHEN 4 THEN 'Financial justification verified'
        ELSE 'No additional notes'
    END,
    DATEADD('day', ABS(MOD(RANDOM(), 18)) + 3, APPLICATION_DATE)::TIMESTAMP_NTZ
FROM RAW_POLICIES;

In [ ]:
%%sql -r create_premiums
CREATE OR REPLACE TABLE RAW_PREMIUMS (
    PREMIUM_ID          VARCHAR(20)     PRIMARY KEY,
    POLICY_ID           VARCHAR(20),
    POLICYHOLDER_ID     VARCHAR(20),
    DUE_DATE            DATE,
    PAYMENT_DATE        DATE,
    AMOUNT_DUE          NUMBER(10,2),
    AMOUNT_PAID         NUMBER(10,2),
    PAYMENT_METHOD      VARCHAR(30),
    PAYMENT_STATUS      VARCHAR(20),
    GRACE_PERIOD_END    DATE,
    LATE_FEE            NUMBER(8,2),
    TRANSACTION_REF     VARCHAR(50),
    CREATED_AT          TIMESTAMP_NTZ
);

In [ ]:
%%sql -r insert_premiums
INSERT INTO RAW_PREMIUMS
WITH
active_policies AS (
    SELECT POLICY_ID, POLICYHOLDER_ID, EFFECTIVE_DATE, ANNUAL_PREMIUM, PAYMENT_FREQUENCY
    FROM RAW_POLICIES
    WHERE POLICY_STATUS IN ('Active', 'Lapsed', 'Expired')
),
payment_periods AS (
    SELECT SEQ4() AS period_num
    FROM TABLE(GENERATOR(ROWCOUNT => 12))
),
combined AS (
    SELECT
        ap.*,
        pp.period_num,
        ROW_NUMBER() OVER (ORDER BY ap.POLICY_ID, pp.period_num) AS rn
    FROM active_policies ap
    CROSS JOIN payment_periods pp
    WHERE pp.period_num < CASE
        WHEN ap.PAYMENT_FREQUENCY = 'Monthly' THEN 12
        WHEN ap.PAYMENT_FREQUENCY = 'Quarterly' THEN 4
        WHEN ap.PAYMENT_FREQUENCY = 'Semi-Annual' THEN 2
        ELSE 1
    END
)
SELECT
    'PRM-' || LPAD(rn::VARCHAR, 8, '0'),
    POLICY_ID,
    POLICYHOLDER_ID,
    DATEADD('month',
        period_num * CASE
            WHEN PAYMENT_FREQUENCY = 'Monthly' THEN 1
            WHEN PAYMENT_FREQUENCY = 'Quarterly' THEN 3
            WHEN PAYMENT_FREQUENCY = 'Semi-Annual' THEN 6
            ELSE 12
        END,
        EFFECTIVE_DATE
    ),
    CASE WHEN ABS(MOD(RANDOM(), 100)) < 88
         THEN DATEADD('day', MOD(RANDOM(), 20) - 5,
              DATEADD('month',
                  period_num * CASE
                      WHEN PAYMENT_FREQUENCY = 'Monthly' THEN 1
                      WHEN PAYMENT_FREQUENCY = 'Quarterly' THEN 3
                      WHEN PAYMENT_FREQUENCY = 'Semi-Annual' THEN 6
                      ELSE 12
                  END,
                  EFFECTIVE_DATE))
         ELSE NULL
    END,
    ROUND(ANNUAL_PREMIUM / CASE
        WHEN PAYMENT_FREQUENCY = 'Monthly' THEN 12
        WHEN PAYMENT_FREQUENCY = 'Quarterly' THEN 4
        WHEN PAYMENT_FREQUENCY = 'Semi-Annual' THEN 2
        ELSE 1
    END, 2),
    CASE WHEN ABS(MOD(RANDOM(), 100)) < 88
         THEN ROUND(ANNUAL_PREMIUM / CASE
            WHEN PAYMENT_FREQUENCY = 'Monthly' THEN 12
            WHEN PAYMENT_FREQUENCY = 'Quarterly' THEN 4
            WHEN PAYMENT_FREQUENCY = 'Semi-Annual' THEN 2
            ELSE 1
         END, 2)
         ELSE 0
    END,
    CASE ABS(MOD(RANDOM(), 5))
        WHEN 0 THEN 'ACH/Direct Debit'
        WHEN 1 THEN 'Credit Card'
        WHEN 2 THEN 'Check'
        WHEN 3 THEN 'Wire Transfer'
        ELSE 'Online Payment'
    END,
    CASE
        WHEN ABS(MOD(RANDOM(), 100)) < 80 THEN 'Paid'
        WHEN ABS(MOD(RANDOM(), 100)) < 88 THEN 'Paid Late'
        WHEN ABS(MOD(RANDOM(), 100)) < 95 THEN 'Overdue'
        ELSE 'Missed'
    END,
    DATEADD('day', 30,
        DATEADD('month',
            period_num * CASE
                WHEN PAYMENT_FREQUENCY = 'Monthly' THEN 1
                WHEN PAYMENT_FREQUENCY = 'Quarterly' THEN 3
                WHEN PAYMENT_FREQUENCY = 'Semi-Annual' THEN 6
                ELSE 12
            END,
            EFFECTIVE_DATE)
    ),
    CASE WHEN ABS(MOD(RANDOM(), 100)) BETWEEN 80 AND 87
         THEN ROUND(ABS(MOD(RANDOM(), 60)) + 15, 2)
         ELSE 0
    END,
    'TXN-' || UUID_STRING(),
    DATEADD('month',
        period_num * CASE
            WHEN PAYMENT_FREQUENCY = 'Monthly' THEN 1
            WHEN PAYMENT_FREQUENCY = 'Quarterly' THEN 3
            WHEN PAYMENT_FREQUENCY = 'Semi-Annual' THEN 6
            ELSE 12
        END,
        EFFECTIVE_DATE
    )::TIMESTAMP_NTZ
FROM combined;

In [ ]:
%%sql -r create_claims
CREATE OR REPLACE TABLE RAW_CLAIMS (
    CLAIM_ID            VARCHAR(20)     PRIMARY KEY,
    POLICY_ID           VARCHAR(20),
    POLICYHOLDER_ID     VARCHAR(20),
    CLAIM_TYPE          VARCHAR(50),
    CLAIM_STATUS        VARCHAR(30),
    DATE_OF_LOSS        DATE,
    DATE_REPORTED       DATE,
    DATE_CLOSED         DATE,
    CLAIM_AMOUNT        NUMBER(12,2),
    APPROVED_AMOUNT     NUMBER(12,2),
    PAID_AMOUNT         NUMBER(12,2),
    CAUSE_OF_CLAIM      VARCHAR(200),
    DIAGNOSIS_CODE      VARCHAR(20),
    HOSPITAL_NAME       VARCHAR(200),
    ATTENDING_PHYSICIAN VARCHAR(100),
    CLAIMANT_NAME       VARCHAR(100),
    CLAIMANT_RELATIONSHIP VARCHAR(50),
    ADJUSTER_ID         VARCHAR(20),
    FRAUD_FLAG          BOOLEAN,
    NOTES               VARCHAR(1000),
    CREATED_AT          TIMESTAMP_NTZ,
    UPDATED_AT          TIMESTAMP_NTZ
);

In [ ]:
%%sql -r insert_claims
INSERT INTO RAW_CLAIMS
WITH
eligible_policies AS (
    SELECT POLICY_ID, POLICYHOLDER_ID, PRODUCT_LINE, EFFECTIVE_DATE, SUM_ASSURED,
           ROW_NUMBER() OVER (ORDER BY RANDOM()) AS policy_rn
    FROM RAW_POLICIES
    WHERE POLICY_STATUS IN ('Active', 'Expired')
    LIMIT 800
),
cause_arrays AS (
    SELECT
        ARRAY_CONSTRUCT('Natural Causes','Accidental Death','Terminal Illness') AS life_causes,
        ARRAY_CONSTRUCT('R99','V99','C80') AS life_codes,
        ARRAY_CONSTRUCT('Hospitalization - Cardiac','Hospitalization - Respiratory','Surgery - Orthopedic',
                        'Emergency Room Visit','Outpatient Procedure','Maternity') AS health_causes,
        ARRAY_CONSTRUCT('I25','J44','M54','R10','Z96','O80') AS health_codes,
        ARRAY_CONSTRUCT('Back Injury','Mental Health','Cardiovascular','Cancer Treatment') AS disability_causes,
        ARRAY_CONSTRUCT('M54','F32','I25','C80') AS disability_codes,
        ARRAY_CONSTRUCT('Cancer Diagnosis','Heart Attack','Stroke') AS ci_causes,
        ARRAY_CONSTRUCT('C80','I21','I63') AS ci_codes,
        ARRAY_CONSTRUCT('City General Hospital','St. Mary Medical Center','Memorial Regional',
                        'University Health System','Providence Medical','Baptist Health',
                        'Mercy Hospital','Cedar Sinai Medical','Northwestern Memorial',
                        'Johns Hopkins Clinic') AS hospitals
)
SELECT
    'CLM-' || LPAD(ep.policy_rn::VARCHAR, 7, '0'),
    ep.POLICY_ID,
    ep.POLICYHOLDER_ID,
    CASE
        WHEN ep.PRODUCT_LINE IN ('Term Life','Whole Life') THEN 'Death Benefit'
        WHEN ep.PRODUCT_LINE = 'Health' THEN 'Medical Expense'
        WHEN ep.PRODUCT_LINE = 'Disability' THEN 'Disability Income'
        ELSE 'Critical Illness Benefit'
    END,
    CASE
        WHEN ABS(MOD(RANDOM(), 100)) < 2 THEN 'Under Investigation'
        WHEN ABS(MOD(RANDOM(), 100)) < 17 THEN 'Open'
        WHEN ABS(MOD(RANDOM(), 100)) < 62 THEN 'Approved'
        WHEN ABS(MOD(RANDOM(), 100)) < 77 THEN 'Paid'
        WHEN ABS(MOD(RANDOM(), 100)) < 90 THEN 'Denied'
        ELSE 'Closed'
    END,
    DATEADD('day', ABS(MOD(RANDOM(), 670)) + 30, ep.EFFECTIVE_DATE),
    DATEADD('day', ABS(MOD(RANDOM(), 670)) + 31 + ABS(MOD(RANDOM(), 14)), ep.EFFECTIVE_DATE),
    CASE WHEN ABS(MOD(RANDOM(), 100)) > 17
         THEN DATEADD('day', ABS(MOD(RANDOM(), 670)) + 45 + ABS(MOD(RANDOM(), 75)), ep.EFFECTIVE_DATE)
         ELSE NULL
    END,
    CASE
        WHEN ep.PRODUCT_LINE IN ('Term Life','Whole Life') THEN ep.SUM_ASSURED
        WHEN ep.PRODUCT_LINE = 'Health' THEN ROUND(ABS(MOD(RANDOM(), 149500)) + 500, 2)
        WHEN ep.PRODUCT_LINE = 'Disability' THEN ROUND(ABS(MOD(RANDOM(), 75000)) + 5000, 2)
        ELSE ROUND(ABS(MOD(RANDOM(), 475000)) + 25000, 2)
    END,
    CASE WHEN ABS(MOD(RANDOM(), 100)) BETWEEN 17 AND 89 THEN
        CASE
            WHEN ep.PRODUCT_LINE IN ('Term Life','Whole Life') THEN ep.SUM_ASSURED
            WHEN ep.PRODUCT_LINE = 'Health' THEN ROUND((ABS(MOD(RANDOM(), 149500)) + 500) * (ABS(MOD(RANDOM(), 40)) + 60) / 100.0, 2)
            WHEN ep.PRODUCT_LINE = 'Disability' THEN ROUND((ABS(MOD(RANDOM(), 75000)) + 5000) * (ABS(MOD(RANDOM(), 30)) + 70) / 100.0, 2)
            ELSE ROUND(ABS(MOD(RANDOM(), 475000)) + 25000, 2)
        END
        ELSE NULL
    END,
    CASE WHEN ABS(MOD(RANDOM(), 100)) BETWEEN 62 AND 76 THEN
        CASE
            WHEN ep.PRODUCT_LINE IN ('Term Life','Whole Life') THEN ep.SUM_ASSURED
            ELSE ROUND((ABS(MOD(RANDOM(), 149500)) + 500) * (ABS(MOD(RANDOM(), 40)) + 60) / 100.0, 2)
        END
        ELSE NULL
    END,
    CASE
        WHEN ep.PRODUCT_LINE IN ('Term Life','Whole Life') THEN ca.life_causes[ABS(MOD(RANDOM(), 3))]::VARCHAR
        WHEN ep.PRODUCT_LINE = 'Health' THEN ca.health_causes[ABS(MOD(RANDOM(), 6))]::VARCHAR
        WHEN ep.PRODUCT_LINE = 'Disability' THEN ca.disability_causes[ABS(MOD(RANDOM(), 4))]::VARCHAR
        ELSE ca.ci_causes[ABS(MOD(RANDOM(), 3))]::VARCHAR
    END,
    CASE
        WHEN ep.PRODUCT_LINE IN ('Term Life','Whole Life') THEN ca.life_codes[ABS(MOD(RANDOM(), 3))]::VARCHAR
        WHEN ep.PRODUCT_LINE = 'Health' THEN ca.health_codes[ABS(MOD(RANDOM(), 6))]::VARCHAR
        WHEN ep.PRODUCT_LINE = 'Disability' THEN ca.disability_codes[ABS(MOD(RANDOM(), 4))]::VARCHAR
        ELSE ca.ci_codes[ABS(MOD(RANDOM(), 3))]::VARCHAR
    END,
    ca.hospitals[ABS(MOD(RANDOM(), 10))]::VARCHAR,
    CASE ABS(MOD(RANDOM(), 8))
        WHEN 0 THEN 'Dr. Smith' WHEN 1 THEN 'Dr. Patel' WHEN 2 THEN 'Dr. Chen'
        WHEN 3 THEN 'Dr. Williams' WHEN 4 THEN 'Dr. Johnson' WHEN 5 THEN 'Dr. Lee'
        WHEN 6 THEN 'Dr. Garcia' ELSE 'Dr. Brown'
    END,
    CASE WHEN ep.PRODUCT_LINE IN ('Term Life','Whole Life') THEN
        CASE ABS(MOD(RANDOM(), 4))
            WHEN 0 THEN 'Sarah' WHEN 1 THEN 'Michael' WHEN 2 THEN 'Emily' ELSE 'David'
        END || ' ' ||
        CASE ABS(MOD(RANDOM(), 4))
            WHEN 0 THEN 'Smith' WHEN 1 THEN 'Johnson' WHEN 2 THEN 'Williams' ELSE 'Brown'
        END
        ELSE NULL
    END,
    CASE WHEN ep.PRODUCT_LINE IN ('Term Life','Whole Life') THEN
        CASE ABS(MOD(RANDOM(), 4))
            WHEN 0 THEN 'Spouse' WHEN 1 THEN 'Child' WHEN 2 THEN 'Parent' ELSE 'Sibling'
        END
        ELSE 'Self'
    END,
    'ADJ-' || LPAD((ABS(MOD(RANDOM(), 30)) + 1)::VARCHAR, 4, '0'),
    CASE WHEN ABS(MOD(RANDOM(), 100)) < 3 THEN TRUE ELSE FALSE END,
    CASE ABS(MOD(RANDOM(), 5))
        WHEN 0 THEN 'All documentation received and verified'
        WHEN 1 THEN 'Waiting on additional medical records'
        WHEN 2 THEN 'Claim processed within SLA'
        WHEN 3 THEN 'Subrogation potential identified'
        ELSE 'Standard processing'
    END,
    DATEADD('day', ABS(MOD(RANDOM(), 670)) + 31, ep.EFFECTIVE_DATE)::TIMESTAMP_NTZ,
    DATEADD('day', -ABS(MOD(RANDOM(), 10)), CURRENT_TIMESTAMP())
FROM eligible_policies ep
CROSS JOIN cause_arrays ca;

In [ ]:
%%sql -r verification
SELECT 'RAW_POLICYHOLDERS' AS TABLE_NAME, COUNT(*) AS ROW_COUNT FROM INSURANCE_UNDERWRITING.RAW.RAW_POLICYHOLDERS
UNION ALL SELECT 'RAW_POLICIES', COUNT(*) FROM INSURANCE_UNDERWRITING.RAW.RAW_POLICIES
UNION ALL SELECT 'RAW_UNDERWRITING_DECISIONS', COUNT(*) FROM INSURANCE_UNDERWRITING.RAW.RAW_UNDERWRITING_DECISIONS
UNION ALL SELECT 'RAW_PREMIUMS', COUNT(*) FROM INSURANCE_UNDERWRITING.RAW.RAW_PREMIUMS
UNION ALL SELECT 'RAW_CLAIMS', COUNT(*) FROM INSURANCE_UNDERWRITING.RAW.RAW_CLAIMS
ORDER BY 1;

In [ ]:
%%sql -r sample_policyholders
SELECT * FROM INSURANCE_UNDERWRITING.RAW.RAW_POLICYHOLDERS LIMIT 10;

In [ ]:
%%sql -r sample_policies
SELECT * FROM INSURANCE_UNDERWRITING.RAW.RAW_POLICIES LIMIT 10;

In [ ]:
%%sql -r sample_uw_decisions
SELECT * FROM INSURANCE_UNDERWRITING.RAW.RAW_UNDERWRITING_DECISIONS LIMIT 10;

In [ ]:
%%sql -r sample_premiums
SELECT * FROM INSURANCE_UNDERWRITING.RAW.RAW_PREMIUMS LIMIT 10;

In [ ]:
%%sql -r sample_claims
SELECT * FROM INSURANCE_UNDERWRITING.RAW.RAW_CLAIMS LIMIT 10;